# AutoSpeed

## I. Getting Started Guide

### 1. AutoSpeed Model Introduction

AutoSpeed is a specialized object detection model in VisionPilot for **closest in-path object (CIPO)** detection, used to support speed control and safe following distance decisions. The implementation is inspired by a YOLO-style one-stage detector and is optimized for real-time operation with a default input size of **$640 \times 640$**.

The network follows a **Backbone => Neck (FPN/PAN-style fusion) => Detection Head** design:

- **AutoSpeed Backbone** : extracts multi-scale features (`p3`, `p4`, `p5`) through progressive downsampling. Unlike a standard block stack, it introduces the **AutoSpeed Context block** in multiple stages to inject global context and improve CIPO localization robustness.

- **AutoSpeed Neck** : upsamples and fuses features across scales using `C3K2` and convolution blocks, combining high-level semantics with lower-level spatial detail before prediction.

- **AutoSpeed Head** : predicts bounding boxes and class logits at three feature scales. It uses **Distribution Focal Loss box regression** and decodes predictions into final boxes during inference.

In the current training/inference configuration, AutoSpeed is built with `num_classes = 4`, with class names in `auto_speed.yaml`: `NA`, `CIPO-1`, `CIPO-2`, and `CIPO-3`. During inference, predictions are post-processed with confidence filtering and NMS, then projected back to the original image coordinates from the letterboxed input.

![](../../Media/AutoSpeed_GIF_2.gif)

### 2. Environment Setup

**If you have done these steps of environment setup in other End-to-End Model Tutorials, please skip this and move forward to the next subsection `3. Download Models`.**

First, we need to clone the repository to your local environment. We will use the official repository from the
Autoware Foundation.

In [ ]:
import os

base_dir = os.getcwd()

# Clone Autoware VisionPilot repo, if not yet done
if not os.path.exists("./autoware_vision_pilot"):
    !git clone https://github.com/autowarefoundation/autoware_vision_pilot.git

The VisionPilot project relies on several key libraries, including **PyTorch** for model execution and
**OpenCV** for image processing.

We will use `pip` to install the required dependencies directly from the `requirements.txt` file provided in 
the `Models` directory.

In [ ]:
# Install required dependencies
%pip install --upgrade pip
%pip install -r autoware_vision_pilot/Models/requirements.txt

# Verify the installation (checking torch as a primary dependency)
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA availability: {torch.cuda.is_available()}")

### 3. Download Models

You can manually download the model weight files, using the links below. Subsequent tutorial sections assume
you already downloaded them and put them inside directory: 
`autoware_vision_pilot/Tutorials/E2E_Models/weights/`.

- [Link to Download Pytorch Model Weights (*.pth)](https://drive.google.com/file/d/1iD-LKf5wSuvf0F5OHVHH3znGEvSIS8LY/view?usp=drive_link)
- [Link to Download ONNX FP32 Moel Height (*.onnx)](https://drive.google.com/file/d/1Zhe8uXPbrPr8cvcwHkl1Hv0877HHbxbB/view?usp=drive_link)

You can also use below script block, which uses `gdown` to download above model weights into such default directory.

In [ ]:
%pip install gdown

import gdown

model_dir = "./weights"
if not os.path.exists(model_dir):
    os.makedirs(model_dir)

metadata = {
    "pth"   : {
        "url"       : "https://drive.google.com/uc?id=1iD-LKf5wSuvf0F5OHVHH3znGEvSIS8LY",
        "filepath"  : os.path.join(base_dir, model_dir, "autospeed.pth")
    },
    "onnx"  : {
        "url"       : "https://drive.google.com/uc?id=1Zhe8uXPbrPr8cvcwHkl1Hv0877HHbxbB",
        "filepath"  : os.path.join(base_dir, model_dir, "autospeed.onnx")
    }
}

for weight_type in metadata:

    url         = metadata[weight_type]["url"]
    filepath    = metadata[weight_type]["filepath"]

    if not os.path.exists(filepath):
        print(f"Downloading AutoSpeed {weight_type} weights...")
        gdown.download(url, filepath, quiet = False)
    else:
        print(f"AutoSpeed {weight_type} weights already exist at {filepath}. Skipping download.")

## II. Quick Test/Inference

Pre-requisites: please ensure that you have completed the above **Environment Setup** first.

### 1. Image Inference

The `image_visualization.py` script facilitates batch processing of images by loading a pre-trained model checkpoint, performing a forward pass on each image, and overlaying detected CIPO bounding boxes onto the original frames.

#### a. Run Batch Image Inference

To run this in any environment, we should navigate to the AutoSpeed visualization directory so the script's internal relative paths resolve correctly.

Key Script Parameters:
- `-p / --model_checkpoint_path`: location of your `.pth` file containing the trained weights.
- `-i / --input_image_dirpath`: directory containing `.png`, `.jpg`, or `.jpeg` files to process.
- `-o / --output_image_dirpath`: destination where visualization files will be saved (named as `[original_id]_data.png`).

In [ ]:
# 1. Path declaration
# Here we use the previously downloaded Pytorch weight file.
# For input and output directories, you can change them to your preferred paths.
# For now we will use the default input and output directories provided in the repository.
CHECKPOINT = metadata["pth"]["filepath"]
INPUT_DIR = os.path.join(
    base_dir,
    "./autoware_vision_pilot/Tutorials/E2E_Models/assets/images/"
 )
OUTPUT_DIR = os.path.join(
    base_dir,
    "./autoware_vision_pilot/Tutorials/E2E_Models/results/autospeed/images/"
 )

# 2. Navigate to visualization directory and execute the script
!cd autoware_vision_pilot/Models/visualizations/AutoSpeed && \
python3 image_visualization.py \
    -p {os.path.abspath(CHECKPOINT_DIR)} \
    -i {os.path.abspath(INPUT_DIR)} \
    -o {os.path.abspath(OUTPUT_DIR)}

#### b. Display Results in Notebook

After running image inference, we can use matplotlib and PIL to visualize AutoSpeed results directly inside this notebook.

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

# Path to the generated results
result_files = sorted([
    f for f in os.listdir(OUTPUT_DIR)
    if f.endswith(".png")
])

if (result_files):
    # Display the first 3 results
    fig, axes = plt.subplots(
        1, 3,
        figsize = (20, 10)
    )
    for i, ax in enumerate(axes):
        if (i < len(result_files)):
            img_path = os.path.join(OUTPUT_DIR, result_files[i])
            img = Image.open(img_path)
            ax.imshow(img)
            ax.set_title(f"Result: {result_files[i]}")
            ax.axis("off")
    plt.tight_layout()
    plt.show()
else:
    print("No result images found. Check your output directory.")

### 2. Video Inference

In this subsection, we apply the AutoSpeed model to video streams. The `video_visualization.py` script processes video files frame-by-frame, runs CIPO detection, and writes an annotated output video.

The pipeline supports both **PyTorch** and **ONNX** model formats. It performs inference on each frame and overlays class-colored bounding boxes on detected in-path objects.

Key script parameters:

- `-p / --model_checkpoint_path`: path to model (`.onnx`, `.pt`, or a directory for PyTorch inference).
- `-i / --video_filepath`: path to the source video file.
- `-o / --output_file`: output path prefix for the annotated video (script appends `.avi`).
- `-v / --vis`: optional flag to preview frames during processing.

Technical details:

- The script uses letterbox-style preprocessing inside the inference backend and maps detections back to original frame coordinates.
- ONNX inference uses ONNX Runtime providers (`CUDAExecutionProvider`, fallback `CPUExecutionProvider`).
- Output is encoded as MJPG and saved as `<output_file>.avi`, while preserving the source FPS and frame size.

In [ ]:
# 1. Path declaration
# Similarly as image inference above:
# - Pytorch weight file (ONXX okay too).
# - Input/Output paths can be changed to preferred.
# - Using defaults for now.
MODEL_PATH = metadata["pth"]["filepath"]
INPUT_PATH = os.path.join(
    base_dir,
    "./autoware_vision_pilot/Tutorials/E2E_Models/assets/videos/highway_normal.mp4"
 )
OUTPUT_BASENAME = os.path.join(
    base_dir,
    "./autoware_vision_pilot/Tutorials/E2E_Models/results/autospeed/videos/highway_normal_output"
 )

# 2. Navigate to visualization directory and execute the script
!cd autoware_vision_pilot/Models/visualizations/AutoSpeed && \
python3 video_visualization.py \
    -p {os.path.abspath(MODEL_PATH)} \
    -i {os.path.abspath(INPUT_PATH)} \
    -o {os.path.abspath(OUTPUT_BASENAME)}

print(f"Saved video: {os.path.abspath(OUTPUT_BASENAME)}.avi")

## III. Model Training

### 1. Model Training on New Data

#### a. Load pre-trained model or load vanilla un-trained model for training from scratch

In this repository, AutoSpeed training is launched from `Models/training/auto_speed_trainer.py`, where the model is built with:

```python
model = AutoSpeedNetwork().build_model(version=args.version, num_classes=4)
```

By default, this means **vanilla training from scratch** (no checkpoint is loaded unless you add it).

- **Train from scratch (current default behavior):** run the trainer as-is.
- **Warm-start / fine-tune from a checkpoint:** use the existing helper `util.load_weight(model, ckpt)` with a small script edit.

Use this pattern inside `train(...)` in `auto_speed_trainer.py` right after model creation:

```python
# Option A: train from scratch
CHECKPOINT_PATH = None

# Option B: fine-tune from a pretrained AutoSpeed checkpoint
CHECKPOINT_PATH = "/absolute/path/to/weights/.pth"

model = AutoSpeedNetwork().build_model(version=args.version, num_classes=4)
if CHECKPOINT_PATH is not None:
    model = util.load_weight(model, CHECKPOINT_PATH)
model.cuda()
```

`util.load_weight` only loads parameters with matching names and shapes, which is useful when architectures are compatible but not strictly identical.

Practical recommendation:
- Start from scratch when your data distribution or class semantics differ significantly.
- Start from checkpoint when you want faster convergence and more stable early training metrics.

#### b. Prepare the training datasets

AutoSpeed training in this repository is currently prepared around the OpenLane conversion pipeline under `Models/data_parsing/AutoSpeed/OpenLane/`.

The converter script reorganizes and converts labels into YOLO format:

```
<class_id> <x_center> <y_center> <width> <height>
```

### i) Expected raw structure before conversion

`converter.py` expects these source folders under your dataset root:

```
<DATASET_DIR>/
├── images/
│   ├── training/
│   └── validation/
└── labels/
    ├── training/
    └── validation/
```

### ii) Run conversion

From repository root:

```bash
python3 Models/data_parsing/AutoSpeed/OpenLane/converter.py \
  -d /absolute/path/to/openlane_autospeed_dataset
```

What it does:
- Moves images from `images/training` → `images/train`, and `images/validation` → `images/val`.
- Converts JSON box labels into YOLO `.txt` labels in `labels/train` and `labels/val`.
- Normalizes using fixed image size assumptions (`1920x1280`) from the converter script.
- Maps class id `4` to `3` during conversion.
- Expands training split by moving most validation samples into training (`expand_training_set(fract=0.25)` keeps ~25% in val).

### iii) Output structure used for training

```
<DATASET_DIR>/
├── images/
│   ├── train/
│   └── val/
└── labels/
    ├── train/
    └── val/
```

### iv) Quick sanity check for converted labels

Use the visualization helper on a converted image:

```bash
python3 Models/data_parsing/AutoSpeed/OpenLane/test_conversion.py \
  -i /absolute/path/to/openlane_autospeed_dataset/images/train/<sample>.jpg
```

This renders converted bounding boxes so you can quickly verify class/box alignment before training.

#### c. How to load dataset

`auto_speed_trainer.py` loads dataset samples from the root path provided by `--dataset` and expects this exact layout:

```
<DATASET_DIR>/
├── images/
│   ├── train/   # training images
│   └── val/     # validation images
└── labels/
    ├── train/   # YOLO txt labels for train
    └── val/     # YOLO txt labels for val
```

The script discovers files from:
- `Path(args.dataset + "/images/train/")` for training
- `Path(args.dataset + "/images/val/")` for validation

For each image path, label path is resolved by replacing `/images/` with `/labels/` and extension with `.txt`.

### Recommended checks before launch

1. Every image in `images/train` and `images/val` has a matching `.txt` in the corresponding labels folder.
2. Each label row has 5 values: `<class_id> <x_center> <y_center> <width> <height>`.
3. All box coordinates are normalized to `[0, 1]`.
4. Class ids match your configured classes in `Models/config/auto_speed.yaml`.

### Launch example (dataset loading from converted root)

From `Models/training/`:

```bash
python3 auto_speed_trainer.py \
  --dataset /absolute/path/to/openlane_autospeed_dataset \
  --input-size 640 \
  --batch-size 32 \
  --epochs 30 \
  --version n
```

If you relabel or change dataset files, delete stale cache files (`images/train.cache`, `images/val.cache`) so labels are re-indexed correctly on the next run.

#### d. How to run training

After dataset conversion/preparation is complete, launch training with `auto_speed_trainer.py`.

##### i) Step 1: Confirm key inputs

- `--dataset`: root path containing `images/train`, `images/val`, `labels/train`, `labels/val`.
- `--version`: model scale (`n`, `s`, `m`, `l`, `x`).
- `--input-size`, `--batch-size`, `--epochs`, and optional `--runs_dir`.

The script loads hyperparameters from `Models/config/auto_speed.yaml`, builds `AutoSpeedNetwork`, profiles FLOPs/params, then starts train/val loops.

##### ii) Step 2: Run training

From repository root:

```bash
cd Models/training
python3 auto_speed_trainer.py \
  --dataset /absolute/path/to/openlane_autospeed_dataset \
  --input-size 640 \
  --batch-size 32 \
  --epochs 30 \
  --version n \
  --runs_dir runs
```

Optional multi-GPU launch (DDP):

```bash
cd Models/training
torchrun --nproc_per_node=2 auto_speed_trainer.py \
  --dataset /absolute/path/to/openlane_autospeed_dataset \
  --input-size 640 \
  --batch-size 32 \
  --epochs 30 \
  --version n \
  --runs_dir runs
```

##### iii) Step 3: What the script does during training

- Builds training and validation loaders from `images/train` and `images/val`.
- Applies augmentations (including mosaic/mixup with scheduled mosaic disable near the end).
- Optimizes box/class/DFL losses with AMP and EMA.
- Runs validation each epoch and reports mAP metrics.
- Saves `last.pt` and best-performing `best.pt` under the run weights directory.

##### iv) Run outputs

Each run is auto-created as `runs/runN/` with:
- `runs/runN/weights/best.pt`
- `runs/runN/weights/last.pt`
- TensorBoard event logs for losses and metrics.

#### e. How to visualize training results

This training pipeline offers two practical result-inspection channels:

1. **TensorBoard logs** for epoch-level losses and validation metrics
2. **Saved checkpoints** (`best.pt`, `last.pt`) under each run directory

##### a) Open TensorBoard

Run from `Models/training/` (or provide full path to runs):

```bash
cd Models/training
tensorboard --logdir runs --port 6006
```

Then open `http://localhost:6006` in your browser.

What you should see in Scalars:
- `Loss/box`, `Loss/cls`, `Loss/dfl`
- `Metrics/mAP`, `Metrics/mAP@50`, `Metrics/Recall`, `Metrics/Precision`

##### b) Inspect checkpoints and use them for inference

During training, checkpoints are saved to:
- `Models/training/runs/runN/weights/best.pt`
- `Models/training/runs/runN/weights/last.pt`

You can plug `best.pt` into the inference scripts from Section II to qualitatively check predictions on images/videos.

Example (image inference expects checkpoint directory containing `best.pt`):

```bash
cd Models/visualizations/AutoSpeed
python3 image_visualization.py \
  -p /absolute/path/to/Models/training/runs/runN/weights \
  -i /absolute/path/to/input_images \
  -o /absolute/path/to/output_images
```

Tip: compare outputs from multiple run folders (`run1`, `run2`, …) to track qualitative improvements across experiments.

### 2. Building an Inference Pipeline

#### a. Using ROS2 to publish an image, run inference and visualize results

TBD.

#### b. Using IceOryx to publish an image, run inference and visualize results

TBD.

#### c. Using Zenoh to publish an image, run inference and visualize results

TBD.

#### d. Using Gstreamer to get an image, run inference and visualize results

TBD.